# Backup JADCO — ensemble fixe 50/50

## Cible commune

« Indice médian de contribution attendue des transitions à la croissance annualisée du loyer effectif des unités à échéance ». Pour chaque unité candidate, `p_i` est la probabilité de transition, `g_i` la croissance annualisée conditionnelle attendue, `c_i = p_i * g_i`, et `P1 = médiane(c_i)`.

Model A (champion gelé) et Model B P1_G1 sont complémentaires par leur construction : A emploie ses baselines historiques globales/médianes figées, tandis que B emploie une occurrence hiérarchique province × mois d’échéance et une croissance hiérarchique provinciale. Ce backup ne réentraîne ni ne modifie ces modèles.

## Règle d’ensemble

`Forecast_backup = 0.50 * Forecast_A + 0.50 * Forecast_B`. Les poids sont fixes et n’ont jamais été optimisés. Les origines historiques sont 31 décembre 2022, 2023 et 2024 ; l’origine 2026 est le 31 décembre 2025.

In [ ]:
from pathlib import Path
import pandas as pd

from src.models.ensemble_50_50 import (
    ENSEMBLE_WEIGHTS,
    backtest_ensemble_50_50,
    estimate_2026_ensemble,
)

lease_path = Path("data/raw/equinoxe_lease_history.csv")
if not lease_path.is_file():
    raise FileNotFoundError(f"Données CRM locales requises : {lease_path.resolve()}")
leases = pd.read_csv(
    lease_path,
    dtype={"sPropCode": "string", "sUnitCode": "string", "sBuilding": "string", "sState": "string"},
)
print("Poids fixes :", dict(ENSEMBLE_WEIGHTS))
print("Lignes d’historique chargées :", len(leases))

## Backtests historiques

Le wrapper appelle les APIs de backtest existantes de Model A et Model B. Les origines sont cutoff-safe ; la cible réalisée est comparée entre modèles et le calcul s’arrête si les cohortes/réalisés ne s’alignent pas. Les résultats ci-dessous ne servent pas à ajuster les poids.

In [ ]:
backtest_rows, comparison_metrics = backtest_ensemble_50_50(leases)
display(backtest_rows)
display(comparison_metrics)

## Forecast 2026

Model A appelle son API finale existante. Model B applique exactement la configuration P1_G1 gelée au cutoff du 31 décembre 2025. Le wrapper ne retourne que des valeurs de portefeuille et des comptes agrégés, jamais de prédictions ou identifiants par unité.

In [ ]:
forecast = estimate_2026_ensemble(leases, asking=None)
print(f"Model A 2026       = {forecast['model_a']:.6f}%")
print(f"Model B P1_G1 2026 = {forecast['model_b']:.6f}%")
print(f"Ensemble 50/50     = {forecast['ensemble_50_50']:.6f}%")
print(f"Origine            = {forecast['prediction_origin']}")
print(f"Candidats Model B  = {forecast['model_b_candidate_count']}")

## Limites

- L’ensemble combine des modèles issus de seulement trois folds historiques ; l’incertitude de généralisation demeure importante.
- La cible est un indice médian de contribution attendue, pas une prévision complète du rent roll, de l’inoccupation ou du revenu net.
- Les règles de maturité des labels, la qualité des dates CRM et la stabilité future des relations restent des risques.
- Aucune donnée externe n’est incorporée ici ; aucune performance observée en 2026 n’est utilisée.
- Les poids restent exactement 50/50, quels que soient les résultats comparatifs.